In [1]:
import random

import torch, os
from TTS.api import TTS

# Get device
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "GPT_XTTS_v2.0_Full_7_5"

model_path = f"/cluster/data/deri/TTS/TTS_CH/trained/{model_name}/"
config_path = f"/cluster/data/deri/TTS/TTS_CH/trained/{model_name}/config.json"
# Init TTS
tts = TTS(
    model_path=model_path,
    config_path=config_path,
    progress_bar=True
).to(device)

 > Using model: xtts


D:\00Projects\00AudioProcessing\TTS\TTS\utils\io.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location=map_location, **kwargs)
GPT2Inference

In [ ]:
import os
conditioning_bpath = "/02Datasets/02 Audio Processing/snf_test/wav"
conditioning_sid = 'ce50fa32-dbc4-4f72-bd5f-5128784a5abc'
conditioning_audios = [
    '1e65bc1316280785b1083ec0f4a3383712fe7f94159671f1bcfd6341e7ce605d',
    '359287ecc8dd219c23f8a02e8822cee145da46acc2872687c18838589ef56ff6',
    '35cf58e4ac663e3cfaf694fd6113828e5fe006b54aa1b5bf13810f6c8f51cfeb',
    '36735e15f0f119ac84c4ab73d8d95a6a3ab6c3f53c8d783bf321a38d10274df2',
    '3b71c15a510ed6963a39151f36b1818311a1cf19232d54588f8856ceb0d1133c'
]

conditioning_paths = [os.path.join(conditioning_bpath, conditioning_sid, f"{audio}.wav") for audio in conditioning_audios]

In [2]:
import os
conditioning_path = "/02Datasets/02 Audio Processing/srf_test/test.wav"

conditioning_paths = [conditioning_path]

In [15]:
opath = "/cluster/data/deri/TTS/TTS_CH_SRF/predictions"
os.makedirs(opath, exist_ok=True)

line = 'Die Geburt des ersten Kindes bringt grosse Veränderungen ins Leben der Eltern. Plötzlich müssen sie eine Doppelrolle bewältigen, die sie vorher nicht kannten: als Frau und Mutter respektive als Mann und Vater. NOISE UND SO DASS ES NID ABBRICHT'
tts.tts_to_file(text=line, speaker_wav=conditioning_paths, language="ch_gr", split_sentences=False, file_path=os.path.join(opath, f'test_audio.wav'))

['Die Geburt des ersten Kindes bringt grosse Veränderungen ins Leben der Eltern. Plötzlich müssen sie eine Doppelrolle bewältigen, die sie vorher nicht kannten: als Frau und Mutter respektive als Mann und Vater. NOISE UND SO DASS ES NID ABBRICHT']
 > Processing time: 16.10442805290222
 > Real-time factor: 0.9572531770716357


'/cluster/data/deri/TTS/TTS_CH_SRF/predictions\\test_audio.wav'

In [4]:
out_file = os.path.join(opath, f'test_audio.wav')

In [5]:
from huggingface_hub import hf_hub_download
device = "cuda"

torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)
ecapa2 = torch.jit.load(model_file, map_location='cuda')
ecapa2.half()

RecursiveScriptModule(
  original_name=Net
  (pre_processing): RecursiveScriptModule(
    original_name=Sequential
    (0): RecursiveScriptModule(original_name=Spectrogram)
    (1): RecursiveScriptModule(original_name=Log)
    (2): RecursiveScriptModule(original_name=Normalize)
  )
  (conv_1): RecursiveScriptModule(
    original_name=Sequential
    (0): RecursiveScriptModule(original_name=Conv2d)
    (1): RecursiveScriptModule(original_name=ReLU)
    (2): RecursiveScriptModule(original_name=BatchNorm2d)
    (3): RecursiveScriptModule(original_name=Conv2d)
    (4): RecursiveScriptModule(original_name=ReLU)
    (5): RecursiveScriptModule(original_name=BatchNorm2d)
    (6): RecursiveScriptModule(original_name=Conv2d)
    (7): RecursiveScriptModule(original_name=ReLU)
    (8): RecursiveScriptModule(original_name=BatchNorm2d)
    (9): RecursiveScriptModule(
      original_name=SEBlock2d
      (avg_pool): RecursiveScriptModule(original_name=AdaptiveAvgPool2d)
      (fc): RecursiveScriptModul

In [6]:
import torchaudio
waveform_c, _ = torchaudio.load(conditioning_path)
embedding_c = ecapa2(waveform_c.to(device))

In [7]:
waveform_o, _ = torchaudio.load(out_file)
embedding_o = ecapa2(waveform_o.to(device))

In [8]:
similarity = torch.nn.functional.cosine_similarity(embedding_o, embedding_c)
embedding_o.shape, embedding_c.shape, similarity

(torch.Size([1, 192]),
 torch.Size([2, 192]),
 tensor([0.1919, 0.1913], device='cuda:0', dtype=torch.float16))

In [ ]:
import pandas as pd
dataspeech_path = "/cluster/data/deri/TTS/TTS_CH/calc_snf_clean_dataset 1.csv"

df = pd.read_csv(dataspeech_path)
#filter where quality is Bad
df = df[df['quality'] != 'Bad']
#remove samples with "bad speech quality",  "very bad speech quality", and "slightly bad speech quality"
df = df[~df['pesq_speech_quality'].isin(["bad speech quality", "very bad speech quality", "slightly bad speech quality"])]
#remove samples with sdr_noise "noisy" or "slightly noisy"
df = df[~df['sdr_noise'].isin(["noisy", "slightly noisy"])]
#remove samples with noise "slightly noisy" and "quite noisy"
df = df[~df['noise'].isin(["slightly noisy", "quite noisy"])]
#remove all speakers with less than 5 samples using the speaker_id column
df = df.groupby('speaker_id').filter(lambda x: len(x) >= 5)
#show number of speakers per dialect
df.groupby('dialect')['speaker_id'].nunique()

In [ ]:
#groupby dialect, then get unique speakers, then sample 4 speakers per dialect
unique_speakers_per_dial = df.groupby('dialect')['speaker_id'].unique()
#sample 4 spaekers per dialect
sampled_speakers_per_dial = unique_speakers_per_dial.apply(lambda x: random.sample(list(x), min(len(x), 4)))
#flatten the list
sampled_speakers = [item for sublist in sampled_speakers_per_dial for item in sublist]
#filter the df with the sampled speakers
randdf = df[df['speaker_id'].isin(sampled_speakers)]
#sample 5 samples per speaker
randdf = randdf.groupby('speaker_id').apply(lambda x: x.sample(5, replace=False)).reset_index(drop=True)
randdf['full_clip_id'] = randdf['speaker_id'] + '/' + randdf['sample_id'] + '.mp3'
randdf

In [ ]:
test_sentence_path = '/02Datasets/02 Audio Processing/snf_test/test.tsv'
test_sentence_df = pd.read_csv(test_sentence_path, sep='\t')
#remove all sample_ids that are in the randdf
test_sentence_df = test_sentence_df[~test_sentence_df['path'].isin(randdf['full_clip_id'])]
#remove all samples with duration < 3
test_sentence_df = test_sentence_df[test_sentence_df['duration'] >= 3]
#get unique sentences
unique_test_sentence_df = test_sentence_df.drop_duplicates(subset='sentence')
#store only sentence column
unique_test_sentence_df_cl = unique_test_sentence_df[['sentence']]
#save to disk
unique_test_sentence_df_cl.to_csv(os.path.join(opath, 'snf_short.tsv'), sep='\t', index=False)

In [ ]:
from tqdm import tqdm
#get list of unique speakers in randdf
LANG_MAP = {
    'ch_be': 'Bern',
    'ch_bs': 'Basel',
    'ch_gr': 'Graubünden',
    'ch_in': 'Innerschweiz',
    'ch_os': 'Ostschweiz',
    'ch_vs': 'Wallis',
    'ch_zh': 'Zürich',
}
LANG_MAP_INV = {v:k for k,v in LANG_MAP.items()}

obpath = f"/cluster/data/deri/TTS/TTS_CH/predictions/"
#save unique_test_sentence_df
unique_test_sentence_df.to_csv(os.path.join(obpath, 'unique_test_sentences.tsv'), sep='\t', index=False)
#save randdf
randdf.to_csv(os.path.join(obpath, 'randdf.tsv'), sep='\t', index=False)
#get first 50 sentences from unique_test_sentence_df
to_gen = unique_test_sentence_df.head(50)

unique_speakers = randdf['speaker_id'].unique()
for sid, speaker in enumerate(unique_speakers):
    speaker_df = randdf[randdf['speaker_id'] == speaker]
    dialect = speaker_df['dialect'].iloc[0]
    dial_tag = LANG_MAP_INV[dialect]
    sample_ids = speaker_df['sample_id'].tolist()
    conditioning_paths = [os.path.join(conditioning_bpath, speaker, f"{audio}.wav") for audio in sample_ids]
    opath = f"/cluster/data/deri/TTS/TTS_CH/predictions/{speaker}"
    os.makedirs(opath, exist_ok=True)
    for idx, row in tqdm(to_gen.iterrows(), total=len(to_gen), desc=f"Speaker {sid}/{len(unique_speakers)}: {speaker}"):
        line = row['sentence']
        tts.tts_to_file(text=line, speaker_wav=conditioning_paths, language=dial_tag, split_sentences=False, file_path=os.path.join(opath, f'sent-{idx}.wav'))

In [ ]:
#load randdf 
import os
import pandas as pd
import shutil

obpath = f"/cluster/data/deri/TTS/TTS_CH/predictions/"
randdf = pd.read_csv(os.path.join(obpath, 'randdf.tsv'), sep='\t')

conditioning_bpath = "/02Datasets/02 Audio Processing/snf_test/wav"
#create a folder wav_filtered that only contains the samples in randdf
for idx, row in randdf.iterrows():
    sid = row['speaker_id']
    sample_id = row['sample_id']
    opath = f"/cluster/data/deri/TTS/TTS_CH/predictions/wav_filtered/{sid}"
    os.makedirs(opath, exist_ok=True)
    #copy for a Windows machine
    shutil.copy(os.path.join(conditioning_bpath, sid, f"{sample_id}.wav"), opath)


In [ ]:
import pandas as pd
import os
gpt_file = '/02Datasets/02 Audio Processing/ChatGPT-Sentences.csv'
#load with delimiter ;
gpt_df = pd.read_csv(gpt_file, delimiter=';')
#add a column with the Sentence length in therms of characters
gpt_df['sentence_length'] = gpt_df['Sentence'].apply(lambda x: len(x))
#remove sentences with more than 250 characters
gpt_df = gpt_df[gpt_df['sentence_length'] <= 250]
#remove sentences with less than 20 characters
gpt_df = gpt_df[gpt_df['sentence_length'] >= 75]
#rename Sentence column to sentence
gpt_df.rename(columns={'Sentence': 'sentence'}, inplace=True)
#sotre sentence column
gpt_df = gpt_df[['sentence']]
#save to disk
obpath = f"/cluster/data/deri/TTS/TTS_CH/predictions/"
gpt_df.to_csv(os.path.join(obpath, 'gpt_sentences.tsv'), sep='\t', index=False)

In [ ]:
human_eval_path = '/02Datasets/02 Audio Processing/Extracted_Evaluation_Data.csv'

human_eval_df = pd.read_csv(human_eval_path)
human_eval_df

In [ ]:
#for each Eval Type do a t-test between all paris of models, given the mean and std for each score type
import numpy as np
from scipy.stats import ttest_ind_from_stats
from itertools import combinations
from collections import defaultdict
obpath = f"/cluster/data/deri/TTS/TTS_CH/predictions/"
#load human eval data

human_eval_path = '/02Datasets/02 Audio Processing/Extracted_Evaluation_Data.csv'
human_eval_df = pd.read_csv(human_eval_path)
#group by Eval Type
grouped = human_eval_df.groupby('Eval Type')
#store the results
results = {}
#iterate over each group, repeat this for each Socre type, note that you are given the mean and the std for each score type
#score types SMOS, CMOS, and Intelligibility

for name, group in grouped:
    for score_type in ['SMOS', 'CMOS', 'Intelligibility']:
        #get all models
        models = group['Model'].unique()
        #make all model pairs with no repetitions (a,b) and (b,a) are the same
        model_pairs = list(combinations(models, 2))
        for comb in model_pairs:
            #get mean and std for each model
            model1_mean = float(group[group['Model'] == comb[0]][f'{score_type} Mean'])
            model2_mean = float(group[group['Model'] == comb[1]][f'{score_type} Mean'])
            model1_std = float(group[group['Model'] == comb[0]][f'{score_type} Std'])
            model2_std = float(group[group['Model'] == comb[1]][f'{score_type} Std'])
            
            #do a t-test
            t, p = ttest_ind_from_stats(mean1=model1_mean, std1=model1_std, nobs1=84, mean2=model2_mean, std2=model2_std, nobs2=84)
            results[f"{name}_{score_type}_{comb[0]}_{comb[1]}"] = float(p) < 0.05


In [ ]:
results

In [ ]:
import torch
import torchaudio
from huggingface_hub import hf_hub_download

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)
ecapa2 = torch.jit.load(model_file, map_location='cuda')
ecapa2.half() # optional, but results in faster inference
audio, sr = torchaudio.load(conditioning_paths[0]) # sample rate of 16 kHz expected

embedding = ecapa2(audio.to('cuda'))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embeddings = []
for conditioning_path in conditioning_paths:
    audio, sr = torchaudio.load(conditioning_path)
    embedding = ecapa2(audio.to('cuda'))
    embeddings.append(embedding.cpu().numpy())
    
#do pairwise cosine similarity
similarity_matrix = cosine_similarity(np.array(embeddings).squeeze())
similarity_matrix